In [4]:
import json
import requests
import base64
import tensorflow as tf
import numpy as np

# 1. Load Kamus Terjemahan (Label Mapping)
# Pastikan path ini benar mengarah ke lokasi file json-mu
with open('label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

print("Daftar Profesi yang Dipahami Model:")
print(list(label_mapping.values()))

# Balikkan kamus: dari "Angka" menjadi "Nama Pekerjaan"
index_to_label = {int(k): v for k, v in label_mapping.items()}

# 2. Teks Resume yang akan diuji
sample_resume = """
John Doe
Senior Information Technology Specialist

Professional Summary:
Highly motivated Information Technology professional with over 8 years of experience in system administration, network engineering, and IT support. Proven expertise in designing, implementing, and maintaining robust IT infrastructure for enterprise-level organizations. Strong background in resolving complex technical issues and optimizing system performance.

Professional Experience:
IT Infrastructure Manager | Tech Solutions Inc. | 2019 - Present
- Led a team of IT technicians to manage daily network operations, ensuring 99.9% system uptime across multiple regional offices.
- Administered Active Directory, Microsoft Exchange, and Office 365 environments for over 500 employees.
- Designed and migrated legacy on-premise servers to cloud-based architecture using AWS and Microsoft Azure.
- Implemented strict cybersecurity protocols, including firewall configurations and intrusion detection systems, reducing security breaches by 100%.

Network Support Engineer | Global Data Corp | 2015 - 2019
- Configured and maintained Cisco routers, switches, and VPN networks.
- Provided Tier 3 technical support for hardware and software issues, consistently exceeding SLA targets.
- Automated routine server maintenance tasks using Python and Bash scripting.

Skills:
- System Administration: Windows Server, Linux (Ubuntu, CentOS), macOS.
- Cloud & Virtualization: AWS, Azure, VMware, Docker.
- Networking: TCP/IP, DNS, DHCP, LAN/WAN, Cisco Networking.
- Security: Cybersecurity best practices, Endpoint protection, Firewalls.
- Scripting & Databases: Python, PowerShell, Bash, SQL (MySQL, Oracle).

Education:
Bachelor of Science in Information Technology, University of Technology, 2015.
Certifications: CompTIA Security+, AWS Certified Solutions Architect, Cisco CCNA.
"""

# 3. Bungkus teks ke dalam format tf.train.Example (Syarat dari TFX)
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value.encode('utf-8')]))

example = tf.train.Example(features=tf.train.Features(feature={
    'resume_text': _bytes_feature(sample_resume)
}))

# Serialize dan ubah ke Base64 agar bisa dikirim via format JSON (HTTP)
serialized_example = example.SerializeToString()
b64_example = base64.b64encode(serialized_example).decode('utf-8')

# 4. Siapkan Paket Data (Payload) untuk dikirim ke Docker
payload = {
    "signature_name": "serving_default",
    "instances": [
        {"examples": {"b64": b64_example}}
    ]
}

url = "http://localhost:8501/v1/models/smart-ats-model:predict"

# 5. Tembak API-nya!
print("Mengirim CV ke Smart ATS Docker...")
try:
    response = requests.post(url, json=payload)
    response.raise_for_status() # Cek jika ada error HTTP
    
    # Ambil array probabilitas dari jawaban Docker
    predictions = response.json()['predictions'][0]

    # Cari nilai probabilitas tertinggi (Kategori yang paling cocok)
    predicted_index = int(np.argmax(predictions))
    confidence = predictions[predicted_index] * 100
    predicted_category = index_to_label[predicted_index]

    print("\n" + "="*40)
    print("🎯 HASIL ANALISIS SMART ATS")
    print("="*40)
    print(f"Kategori Profesi : {predicted_category.upper()}")
    print(f"Tingkat Keyakinan: {confidence:.2f}%")
    print("="*40)

except Exception as e:
    print(f"Gagal menghubungi server: {e}")

Daftar Profesi yang Dipahami Model:
['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
Mengirim CV ke Smart ATS Docker...

🎯 HASIL ANALISIS SMART ATS
Kategori Profesi : INFORMATION-TECHNOLOGY
Tingkat Keyakinan: 94.59%
